In [1]:
%load_ext autoreload
%autoreload 2
%xmode verbose

Exception reporting mode: Verbose


In [17]:
from Exoplanet import Exoplanet

data_paths = []

with open('Exoplanet Discovering/candidate_data_paths.txt', 'r+') as file:
    data_paths.extend([file_name.strip() for file_name in file.readlines()])
    file.seek(0)

for path in data_paths:
    e = Exoplanet.load_from_json(path)

    e.calculate_attributes() # update parametes
    e.make_csv_string()

    print(f'{e.id_num}: {e.mass_unc_higher}')

    e.to_json(f'Exoplanet Discovering/Paper Materials/Data/{e.host_star['name']} Exoplanet {e.letter}.json')

with open('Exoplanet Discovering/candidate_data_paths.txt', 'r+') as file:
    file.truncate(0)

11799872: 628.5672535825876
23689332: 636.9703673504198
35655828: 5187.2798130138635
35655828: 183.72063752291945
38850860: 780.1846078357414
55655482: 680.575692554152
55727842: 24.863608854541187
62573638: 996.7036786748466
73248090: 2246.5908193312216
76903841: 10643.81030477944
90083037: 1018.6186960948912
102625324: 647.6090783582489
176871438: 1259.3010888376539
234518605: 1146.9369126044094
268290940: 73.36397801735217
270462117: 843.2668151399021
279322914: 1049.905607852655
284611896: nan
295440174: 1678.5622619179017
297701617: 5486.196789075775
300291839: 102.21466767192445
306635088: 1668.6726963798037
342556321: 1525.0127902280778
375477302: 744.7968752669378
441444256: nan
451596415: 115.11254791322924
467113954: 1751.591649333196
734505586: 70.48434952643228


In [ ]:
from Exoplanet import Exoplanet
from astroquery.mast import Catalogs
import lightkurve
from lightkurve.correctors import DesignMatrix, RegressionCorrector
import numpy

data_paths = []

with open('Exoplanet Discovering/candidate_data_paths.txt', 'r+') as file:
    data_paths.extend([file_name.strip() for file_name in file.readlines()])
    file.seek(0)

for path in data_paths:
    e = Exoplanet.load_from_json(path)

    data = Catalogs.query_object(f'TIC {e.id_num}', catalog = 'TIC')[0]

    host_star = {
        'radius': data['rad'],
        'mass': data['mass'],
        'lum': data['lum'],
        'teff': data['Teff'],
        'logg': data['logg'],
        'ra': data['ra'],
        'dec': data['dec']
    }

    e.add_host_star_attributes(**host_star)

    search_result = lightkurve.search_targetpixelfile(e.host_star['name'], mission = 'TESS', cadence = 'long')
    pixel_files = search_result.download_all(limit = 20)

    collection = lightkurve.LightCurveCollection(None)
    for pixel_file in pixel_files[:]:
        pixel_file = pixel_file[numpy.isfinite(pixel_file.flux).all(axis = (1, 2))]
        pixel_file = pixel_file[numpy.isfinite(pixel_file.flux_err).all(axis = (1, 2))]

        aperture_mask = pixel_file.pipeline_mask
        uncorrected_lc = pixel_file.to_lightcurve(aperture_mask = aperture_mask)

        design_matrix = DesignMatrix(pixel_file.flux[:, ~aperture_mask]).pca(5).append_constant()
        lc = RegressionCorrector(uncorrected_lc).correct(design_matrix)
        lc = uncorrected_lc
        lc = lc.normalize().flatten(niters = 10).remove_outliers(10.0)
        collection.append(lc)

    lc = collection.stitch()
    lc = lc.flatten()

    e.calculate_attributes()
    e.calculate_transit_model_params(lc)
    e.make_csv_string()

    print(f'{e.id_num}: {e.impact_parameter}')

    e.to_json(f'Exoplanet Discovering/Paper Materials/Data/{e.host_star['name']} Exoplanet {e.letter}.json')

with open('Exoplanet Discovering/candidate_data_paths.txt', 'r+') as file:
    file.truncate(0)

In [18]:
import os
import json

data = []

for (dirpath, dirnames, filenames) in os.walk('Exoplanet Discovering/Paper Materials/Data'):
    for filepath in filenames:
        if filepath == '.DS_Store' or filepath.endswith('.csv'):
            continue

        with open(dirpath + '/' + filepath, 'r') as file:
            exoplanet = json.loads(file.read())

        if type(exoplanet) == list:
            continue

        data.append(exoplanet)

    break

data = sorted(data, key = lambda x: x['name'])

with open('Exoplanet Discovering/Paper Materials/Data/full.json', 'w') as file:
    string = json.dumps(data, indent = 4)
    string = string.replace('NaN', 'null')
    string = string.replace('nan', '')

    file.write(string)

In [ ]:
import json

with open('Exoplanet Discovering/Paper Materials/Data/full.json', 'r') as file:
    data = json.loads(file.read())

with open('Exoplanet Discovering/Paper Materials/params_planet_20250504_001.txt', 'w') as file:
    string = ''
    for exoplanet in data:
        string += exoplanet['csv_string'] + '\n'

    file.write(string)

In [21]:
import json
import pandas
import math

with open('Exoplanet Discovering/Paper Materials/Data/full.json', 'r') as file:
    data = json.loads(file.read())

data = pandas.DataFrame([exoplanet['parameters'] for exoplanet in data])
data['target'] = data['target'].apply(lambda x: x.replace('TIC', 'TIC '))
data = data.set_index('target')
data.replace('', math.nan, inplace = True)
data = data.drop(['tag', 'flag'], axis = 1)
data['disp'] = data['disp'].mask(data['radius'] >= 23.53885947 + 1, 'FP') # z-score radius limit of all candidates/planets, from Nasa Exoplanet Archive
data = data.sort_values(['disp', 'target'
                         ], ascending = [False, True])

data = data.drop(['TIC ' + str(ticid) + '.01' for ticid in [
        284611896,
        268290940,
        62573638,
        23689332,
        441444256,
        234518605,
        306635088,
        451596415,
        375477302]])

data = data[['disp', 'period', 'period_unc', 'epoch', 'epoch_unc', 'r_planet', 'radius', 'mass', 'mass_unc_lower', 'mass_unc_higher', 'temp', 'depth', 'depth_unc', 'duration', 'duration_unc', 'sma', 'imp', 'inc', 'arg_peri', 'ecc', 'u1', 'u2']]
data['epoch'] -= 2457000
columns = data.select_dtypes(include = 'number').columns.to_list()

for value in ['period', 'depth', 'duration', 'epoch']:
    columns.remove(value)

data[columns] = data[columns].map(
    lambda x: x if x == 0 or math.isnan(x) else round(x, (3 - 1) - int(math.floor(math.log10(abs(x)))))
)

data[['period', 'depth', 'duration', 'epoch']] = data[['period', 'depth', 'duration', 'epoch']].map(
    lambda x: x if x == 0 or math.isnan(x) else round(x, (6 - 1) - int(math.floor(math.log10(abs(x)))))
)

data.replace(math.nan, 'null', inplace = True)

data.index.names = ['Target (CTOI ID)']
data.index = data.index.str.replace('TIC ', '', regex = False)
data['Period (days)'] = data['period'].astype(str) + ' ± ' + data['period_unc'].astype(str)
data['Epoch (BJD_TDB - 2457000)'] = data['epoch'].astype(str) + ' ± ' + data['epoch_unc'].astype(str)
data['Transit Depth (PPM)'] = data['depth'].astype(str) + ' ± ' + data['depth_unc'].astype(str)
data['Transit Duration (hours)'] = data['duration'].astype(str) + ' ± ' + data['duration_unc'].astype(str)

data['Disposition'] = data['disp']
data['Planet Radius / Stellar Radius'] = data['r_planet']
data['Radius (R⊕)'] = data['radius']
data['Mass (M⊕)'] = data['mass']
data['Mass Unc (M⊕)'] = '+' + data['mass_unc_higher'].astype(str) + ', ' + data['mass_unc_lower'].astype(str)
data['Equilibrium Temperature (K)'] = data['temp']
data['Semi-major Axis (AU)'] = data['sma']

data['Impact Parameter'] = data['imp']
data['Inclination (deg)'] = data['inc']
data['Eccentricity'] = data['ecc']
data['Argument of Periastron'] = data['arg_peri']
data['Limb Darkening Coefficient 1'] = data['u1']
data['Limb Darkening Coefficient 2'] = data['u2']

data = data[['Period (days)', 'Epoch (BJD_TDB - 2457000)', 'Transit Depth (PPM)', 'Transit Duration (hours)',
             'Disposition', 'Planet Radius / Stellar Radius', 'Radius (R⊕)', 'Mass (M⊕)', 'Mass Unc (M⊕)', 'Equilibrium Temperature (K)', 'Semi-major Axis (AU)',
             'Impact Parameter', 'Inclination (deg)', 'Eccentricity', 'Argument of Periastron', 'Limb Darkening Coefficient 1', 'Limb Darkening Coefficient 2']]

data.to_csv('Exoplanet Discovering/Paper Materials/ctois.csv')

In [ ]:
import json
import pandas
import math

with open('Exoplanet Discovering/Paper Materials/Data/full.json', 'r') as file:
    data = json.loads(file.read())

data = pandas.DataFrame([exoplanet['host_star'] for exoplanet in data])
data = data.set_index('name')
data.replace('', math.nan, inplace = True)
data = data.sort_index()

data = data.drop(['TIC ' + str(ticid) for ticid in [
        284611896,
        268290940,
        62573638,
        23689332,
        441444256,
        234518605,
        306635088,
        451596415,
        375477302]])

data = data[['dec', 'logg', 'lum', 'mass', 'ra', 'radius', 'teff']]
columns = data.select_dtypes(include = 'number').columns.to_list()

for value in ['ra', 'dec']:
    columns.remove(value)

data[columns] = data[columns].map(
    lambda x: x if x == 0 or math.isnan(x) else round(x, (3 - 1) - int(math.floor(math.log10(abs(x)))))
)

data[['ra', 'dec']] = data[['ra', 'dec']].map(
    lambda x: x if x == 0 or math.isnan(x) else round(x, (6 - 1) - int(math.floor(math.log10(abs(x)))))
)

data.replace(math.nan, 'null', inplace = True)

data.index.names = ['Target (TIC ID)']
data.index = data.index.str.replace('TIC ', '', regex = False)

data['Declination (deg)'] = data['dec']
data['Right Ascension (deg)'] = data['ra']
data['Radius (R⊙)'] = data['radius']
data['Mass (M⊙)'] = data['mass']
data['Surface Gravity (log(g))'] = data['logg']
data['Effective Temperature (K)'] = data['teff']

data = data[['Declination (deg)', 'Right Ascension (deg)', 'Radius (R⊙)', 'Mass (M⊙)', 'Surface Gravity (log(g))', 'Effective Temperature (K)']]

data.to_csv('Exoplanet Discovering/Paper Materials/stars.csv')

In [ ]:
import pandas
import json

with open('Exoplanet Discovering/Paper Materials/Data/full.json', 'r') as file:
    data = json.loads(file.read())

ticids = [exoplanet['id_num'] for exoplanet in data]

data = pandas.DataFrame()
for ticid in ticids:
    url = f'https://exofop.ipac.caltech.edu/tess/download_planet.php?id={ticid}'

    data = pandas.concat([data, pandas.read_csv(url, delimiter = '|')])

data = data.set_index('Name')
data = data.sort_index()
data

In [ ]:
from Exoplanet import Exoplanet
import numpy
import lightkurve
import matplotlib.pyplot as plot
from Exoplanet import Exoplanet
import numpy
from astropy.time import TimeDelta
from lightkurve.correctors import DesignMatrix, RegressionCorrector

e = Exoplanet.load_from_json('Exoplanet Discovering/Paper Materials/Data/TIC 55727842 Exoplanet b.json')

search_result = lightkurve.search_targetpixelfile(e.host_star['name'], mission = 'TESS', cadence = 'long')
pixel_files = search_result.download_all(limit = 20)

collection = lightkurve.LightCurveCollection(None)
for pixel_file in pixel_files[:]:
    pixel_file = pixel_file[numpy.isfinite(pixel_file.flux).all(axis = (1, 2))]
    pixel_file = pixel_file[numpy.isfinite(pixel_file.flux_err).all(axis = (1, 2))]

    aperture_mask = pixel_file.pipeline_mask
    # aperture_mask = pixel_file.create_threshold_mask(threshold = 10, reference_pixel = 'center')
    uncorrected_lc = pixel_file.to_lightcurve(aperture_mask = aperture_mask)

    design_matrix = DesignMatrix(pixel_file.flux[:, ~aperture_mask]).pca(5).append_constant()
    lc = RegressionCorrector(uncorrected_lc).correct(design_matrix)
    lc = uncorrected_lc
    lc = lc.normalize().flatten(niters = 10).remove_outliers(10.0)
    collection.append(lc)

lc = collection.stitch()
lc = lc.flatten()

In [ ]:
import batman

params = e.calculate_transit_model_params(lc)
time = lc.time.value
# lc = lc.fold(period = e.period, t0 = e.transit_time)
# lc.scatter()
full_params = [e.period, e.transit_time, e.r_planet_over_star, e.sma_over_r_star]
full_params.extend(params)

model = Exoplanet.transit_model(full_params, time)

plot.plot(time, model)
plot.show()